In [ ]:
import os
import torch
from utils import load_checkpoint_for_eval
from table_matching_model import strubert_embd
import numpy as np

## Creating the dataset for processing SPIDER dataset
### Processing the query tables

In [ ]:
m_parm_setting = f'default_model'
parm_setting = f'no_aug'
# model_setting = "batch_4_epoch_8_lr_3e-05"

query_path_output = "../../dataset/spider/processing/query"

result_path = f"../../output/spider_res/default_{m_parm_setting}_{parm_setting}/"

corpus_path_output = f"../../dataset/spider/processing/corpus/"

corpus_folder = corpus_path_output
query_folder = query_path_output

## Loading the Model

In [ ]:
# Load the best model from the database

tabert_path = "../../model/tabert_base_k3/model.bin"
device = "cuda:3"

In [ ]:
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())
print("Current CUDA Device:", torch.cuda.current_device())

In [ ]:
# Loading the model
model = strubert_embd(tabert_path=tabert_path, device=device) 

In [ ]:
# Assume this is your loaded model

model.eval()


## Reading the dataset to generate embedding

In [ ]:
import os
import json

# creating the list of all the tables

# input will be a list of list [['table-0312-73' 'table-0312-60' '1']]
corpus = []

# List all files
for filename in os.listdir(corpus_folder):
    file_path = os.path.join(corpus_folder, filename)
    if os.path.isfile(file_path):
        print("Reading:", file_path)
        with open(file_path, "r") as f:
            data = json.load(f)
            data_keys = list(data.keys())
            # print(data_keys)
            for k in data_keys:
                corpus.append([k,k,1])

In [ ]:
print(len(corpus))

# corpus = corpus[:30]

In [ ]:
from data_reader import DataAndQueryReader
from torch.utils.data import DataLoader
from utils import pad_table_query

batch_size = 5

corpus_dataset = DataAndQueryReader(corpus,corpus_folder)
test_iter = DataLoader(corpus_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_table_query)


In [ ]:
corpus_embddding = []

for i, batch in enumerate(test_iter):
    tables, queries, all_tables_meta, all_query_meta, labels = batch

    labels = torch.FloatTensor(labels).to(device)
    
    with torch.no_grad():  # ensures no gradients are tracked
        outputs = model(tables, all_tables_meta, queries, all_query_meta)

    # detach and move to CPU before storing
    corpus_embddding.append(outputs)

In [ ]:
print(len(corpus_embddding))
print(corpus_embddding[-1][0].shape[0])

### Reading the query table and generate its embedding

In [ ]:
import os
import json

# creating the list of all the tables

# input will be a list of list [['table-0312-73' 'table-0312-60' '1']]
query = []

# List all files
for filename in os.listdir(query_folder):
    file_path = os.path.join(query_folder, filename)
    if os.path.isfile(file_path):
        print("Reading:", file_path)
        with open(file_path, "r") as f:
            data = json.load(f)
            data_keys = list(data.keys())
            # print(data_keys)
            for k in data_keys:
                for _ in range(batch_size):
                    query.append([k,k,1])

In [ ]:
print(len(query))

# query = query[:50]
# query

In [ ]:
from data_reader import DataAndQueryReader
from torch.utils.data import DataLoader
from utils import pad_table_query


query_dataset = DataAndQueryReader(query,query_folder)

query_iter = DataLoader(query_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_table_query)

In [ ]:
query_embddding = []

for i, batch in enumerate(query_iter):
    tables, queries, all_tables_meta, all_query_meta, labels = batch

    labels = torch.FloatTensor(labels).to(device)
    
    with torch.no_grad():  # ensures no gradients are tracked
        outputs = model(tables, all_tables_meta, queries, all_query_meta)

    # detach and move to CPU before storing
    query_embddding.append(outputs)

In [ ]:
print(query_embddding[-1][0][:4].shape)

### Get the similarity for query with corpus

In [ ]:

res = []

for q in range(len(query_embddding)):
    r = []
    # print(q)
    for c in range(len(corpus_embddding)):
        # print(c)
        with torch.no_grad():  # ensures no gradients are tracked
            # rnk = model.get_similarity_custom(query_embddding[q][0][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][1][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][2][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][3][:min(batch_size, corpus_embddding[c][0].shape[0])], corpus_embddding[c][0], corpus_embddding[c][1], corpus_embddding[c][2], corpus_embddding[c][3])
            rnk = model.get_similarity_cls(query_embddding[q][0][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][1][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][2][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][3][:min(batch_size, corpus_embddding[c][0].shape[0])], corpus_embddding[c][0], corpus_embddding[c][1], corpus_embddding[c][2], corpus_embddding[c][3])
        r.append(rnk)
    res.append(r)


In [ ]:
len(res[0][0])

In [ ]:
print(res[0])

In [ ]:
corpus_list = []

for c in corpus:
    corpus_list.append(c[0])

In [ ]:
print(len(corpus_list))

In [ ]:
query_list = []
for q in query:
    if q[0] not in query_list:
        query_list.append(q[0])
    

In [ ]:
print(len(query_list))

In [ ]:
# checking if the folder exist in that path

if not os.path.exists(result_path):
    os.makedirs(result_path)
    print("folder created...")
else:
    print("folder already exist...")

In [ ]:
from torch import nn
m = nn.Sigmoid()

for idx, r in enumerate(res):
    out = torch.cat(r, dim=0)
    out_s = m(out)

    res_dic = {}

    for o in range(out_s.shape[0]):
        if out_s[o].item() not in res_dic.keys():
            res_dic[out_s[o].item()] = []
        res_dic[out_s[o].item()].append(o)

    res_dic_key = list(res_dic.keys())
    sorted_res_dic_key = sorted(res_dic_key, reverse=True)
    
    with open(os.path.join(result_path, query_list[idx]), 'w') as fp:
        for k in sorted_res_dic_key:
            fp.write(f'{k} : ')
            for j in res_dic[k]:
                fp.write(f'{corpus_list[j]} ')
            
            fp.write('\n')